# Shapley_cal.ipynb — 부분집합 v(S) 분산 계산 워커

`simulation_external_excute`(repo #1)의 AWS 파이프라인이 **각 EC2 인스턴스에서** 실행하는 노트북.

- `.env`의 `SHAPLEY_SUBSETS`(이 인스턴스가 맡은 부분집합 목록)를 읽어 각 부분집합의
  `v(S) = min_w J(w; S)` 를 PSO inner-loop로 계산한다.
- 결과를 `tuning/shapley_results/part_<SHAPLEY_TAG>.json` 으로 저장한다
  (`composite_rule.ipynb`의 `value_cache.json` 과 동일한 `entries` 포맷).
- `run_simulation.sh` 가 이 part json 을 Google Drive 로 업로드한다.
- 모든 인스턴스의 part json 을 `tuning/shapley_merge.py` 로 병합 → `value_cache.json`.

기본 설정(`N_RUNS=10`, PSO `swarm=4/n_iter=5/seed=0`, `base_seed=0`,
`DOWN_ACTIVE=True`, `PM_ACTIVE=False`)은 기존 `value_cache.json` 의 meta 와 일치하도록
맞춰 결과가 같은 캐시로 풀링 가능하다. `.env` 로 덮어쓸 수 있다.


In [ ]:
import os
import sys
import json
import time

import numpy as np
from dotenv import load_dotenv

# nbconvert 는 노트북 디렉토리(tuning/)에서 실행 → PROJECT_ROOT 는 그 상위
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

load_dotenv(os.path.join(PROJECT_ROOT, '.env'), override=True)

from utils import DataLoader
from tuning.saa_evaluator import SAAEvaluator, TRANSITION_RISK_CACHE_PATH
from tuning.composite_rule import (
    load_normalization_ref,
    DynamicCompositeEvaluator,
    value_function,
)
from simulation.priority import TERM_REGISTRY, available_terms

print('PROJECT_ROOT =', PROJECT_ROOT)


## 1. 이 인스턴스가 맡은 부분집합 + 설정 (.env)

`SHAPLEY_SUBSETS` 포맷: 세미콜론(`;`)으로 부분집합 구분, 콤마(`,`)로 항 구분.
빈 집합(FIFO baseline `v(∅)`)은 `EMPTY` 토큰으로 표현.

예) `SHAPLEY_SUBSETS=PT,SLACK ; C_TRANSITION,SETUP,WAITING ; EMPTY`


In [ ]:
def _getenv_int(key, default):
    v = os.environ.get(key, '').strip()
    return int(v) if v else default

def _getenv_bool(key, default):
    v = os.environ.get(key, '').strip().lower()
    if not v:
        return default
    return v in ('1', 'true', 'yes', 'y', 't')

# --- 이 인스턴스가 맡은 부분집합 파싱 ---
raw = os.environ.get('SHAPLEY_SUBSETS', '').strip()
if not raw:
    raise SystemExit('ERROR: .env 에 SHAPLEY_SUBSETS 가 설정되지 않았습니다.')

SUBSETS = []
for seg in raw.split(';'):
    seg = seg.strip()
    if not seg:
        continue
    if seg.upper() == 'EMPTY':
        SUBSETS.append([])  # v(∅) = FIFO baseline
        continue
    terms = sorted(t.strip().upper() for t in seg.split(',') if t.strip())
    # 항 검증
    for t in terms:
        if t not in TERM_REGISTRY:
            raise SystemExit(f'ERROR: 등록되지 않은 항 {t!r} (사용 가능: {available_terms()})')
    SUBSETS.append(terms)

TAG = os.environ.get('SHAPLEY_TAG', '').strip() or 'untagged'

# --- 평가 설정 (value_cache.json meta 와 일치하는 기본값) ---
N_RUNS    = _getenv_int('SHAPLEY_N_RUNS', 10)
SWARM     = _getenv_int('SHAPLEY_PSO_SWARM', 4)
N_ITER    = _getenv_int('SHAPLEY_PSO_NITER', 5)
PSO_SEED  = _getenv_int('SHAPLEY_PSO_SEED', 0)
BASE_SEED = _getenv_int('SHAPLEY_BASE_SEED', 0)
DOWN_ACTIVE = _getenv_bool('DOWN_ACTIVE', True)
PM_ACTIVE   = _getenv_bool('PM_ACTIVE', False)
PSO_KWARGS  = dict(swarm_size=SWARM, n_iter=N_ITER, seed=PSO_SEED)

print(f'TAG={TAG}  맡은 부분집합 {len(SUBSETS)}개:')
for s in SUBSETS:
    print('   ', s if s else '∅ (FIFO)')
print(f'설정: N_RUNS={N_RUNS}, PSO={PSO_KWARGS}, base_seed={BASE_SEED}, '
      f'DOWN_ACTIVE={DOWN_ACTIVE}, PM_ACTIVE={PM_ACTIVE}')


## 2. 데이터 / 정규화 기준 / 평가기 초기화

In [ ]:
# 데이터 로드 (ansible 이 CSV 를 PROJECT_ROOT/<BASE_DATA_PATH> 로 복사)
base_data_path = os.environ.get('BASE_DATA_PATH', 'data').strip()
if not os.path.isabs(base_data_path):
    base_data_path = os.path.join(PROJECT_ROOT, base_data_path)
data = DataLoader(base_data_path).load_all_data()
print(f'데이터 로드: jobs={len(data["jobs"])}, machines={len(data["machines"])}  ({base_data_path})')

# 정규화 기준 (repo 에 tracked: data/normalization_ref.json)
NORM_REF_PATH = os.path.join(PROJECT_ROOT, 'data', 'normalization_ref.json')
MK_REF, QV_REF_AQT, _ = load_normalization_ref(NORM_REF_PATH)
print(f'mk_ref={MK_REF:.3f}, qv_ref_aqt={QV_REF_AQT:.3f}')

# transition_risk 캐시 (없으면 SAAEvaluator 가 deterministic 하게 생성)
if not os.path.exists(TRANSITION_RISK_CACHE_PATH):
    print('transition_risk 캐시 없음 → SAAEvaluator 로 생성')
    SAAEvaluator(data, obj_weights=(1.0, 1.0), base_seed=BASE_SEED,
                 down_active=DOWN_ACTIVE, pm_active=PM_ACTIVE,
                 mk_ref=MK_REF, qv_ref=QV_REF_AQT)

evaluator = DynamicCompositeEvaluator(
    data, mk_ref=MK_REF, qv_ref_aqt=QV_REF_AQT,
    base_seed=BASE_SEED, down_active=DOWN_ACTIVE, pm_active=PM_ACTIVE,
)
print(evaluator.summary())


## 3. 맡은 부분집합들의 v(S) 계산 → part json 저장

In [ ]:
entries = []
t_all = time.time()
for i, S in enumerate(SUBSETS):
    t0 = time.time()
    out = value_function(evaluator, S, N_RUNS, PSO_KWARGS)
    w = [float(x) for x in (out['w*'] if hasattr(out['w*'], '__iter__') else [])]
    entries.append({'terms': list(S), 'J*': float(out['J*']), 'w*': w})
    label = ','.join(S) if S else 'EMPTY(FIFO)'
    print(f'  [{i+1}/{len(SUBSETS)}] {label:<45s} v(S)={out["J*"]:.5f}  '
          f'({time.time()-t0:.1f}s)')

elapsed = time.time() - t_all
print(f'\n총 {len(entries)}개 부분집합 완료, {elapsed:.1f}s')


In [ ]:
# part json 저장 (run_simulation.sh 가 Google Drive 로 업로드)
out_dir = os.path.join(PROJECT_ROOT, 'tuning', 'shapley_results')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, f'part_{TAG}.json')

payload = {
    'meta': {
        'tag': TAG,
        'n_runs': N_RUNS,
        'pso_kwargs': PSO_KWARGS,
        'base_seed': BASE_SEED,
        'down_active': DOWN_ACTIVE,
        'pm_active': PM_ACTIVE,
        'mk_ref': MK_REF,
        'qv_ref_aqt': QV_REF_AQT,
        'elapsed_sec': elapsed,
    },
    'entries': entries,
}
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
print(f'저장 완료: {out_path}  ({len(entries)}개 entry)')
